# Environment check

Run this notebook first, on a Colab runtime with a GPU (the runtime that will
extract and train). It reads the original outputs and writes only to the new
root (`config.ROOT`): folders of the root and small check tables. Every check
is a hard stop.

What it does, in order:

1. mounts Drive, clones this repository at a fixed commit and installs the pinned libraries (on a fresh runtime everything is installed in one pass and the cell stops once: restart the session and run the notebook again from the top);
2. prints the runtime (library versions, text encoding, GPU, free space);
3. creates the folders of the new root (**first write to Drive**) and reads `legacy_map.yaml`;
4. clones the original analysis code (read only) for the parity tests;
5. unpacks ASVspoof 2019 LA from the original archive to the local disk and checks the file lists;
6. runs the whole test suite on this runtime, with the parity tests, the tandem t-DCF reference and the audio and trimming parity on real files;
7. reads what the original feature shards record about audio libraries, and the original sampling rates;
8. reads the commit of the In-the-Wild repository so that it can be pinned.

In [ ]:
from google.colab import drive, userdata
import os

drive.mount("/content/drive")
assert os.path.ismount("/content/drive"), "Google Drive is not mounted"

### Repository at a fixed commit

Set `COMMIT` to the commit of this repository that is being checked: the
full hash or at least its first 7 characters. The repository is public and
is cloned without a token. If the Colab secret `GITHUB_TOKEN` exists and
this notebook has access to it, the clone uses it (a repository that is not
public needs one); the token is not kept in the clone and is replaced by
`***` in everything git prints.

In [ ]:
import subprocess

REPOSITORY = "Nooralabd/silence-shortcut-wavlm-layers"
COMMIT = ""          # the commit to run: the full hash or at least its first 7 characters
WORKDIR = "/content/silence-shortcut-wavlm-layers"

assert len(COMMIT) >= 7 and set(COMMIT) <= set("0123456789abcdef"), \
    "set COMMIT to the commit that is being checked (full hash or at least its first 7 characters)"
try:
    token = userdata.get("GITHUB_TOKEN") or ""
except Exception:    # the secret does not exist, or this notebook has no access to it
    token = ""       # a public repository is cloned without a token


def git(*arguments):
    """Run git and print what it prints; the token is never printed."""
    result = subprocess.run(["git", *arguments], capture_output=True, text=True)
    output = (result.stdout + result.stderr).strip()
    if token:
        output = output.replace(token, "***")
    if output:
        print(output)
    assert result.returncode == 0, "git failed: see the lines above"
    return output


def clone(repository, folder):
    """Clone a repository, or fetch into an existing clone; a token, if there is one, is not kept in it."""
    address = f"https://{token}@github.com/{repository}.git" if token else f"https://github.com/{repository}.git"
    if not os.path.isdir(folder):
        git("clone", "-q", address, folder)
        git("-C", folder, "remote", "set-url", "origin", f"https://github.com/{repository}.git")
    else:
        git("-C", folder, "fetch", "-q", address, "+refs/heads/*:refs/remotes/origin/*")


clone(REPOSITORY, WORKDIR)
git("-C", WORKDIR, "checkout", "-q", COMMIT)
git("-C", WORKDIR, "rev-parse", "HEAD")
%cd {WORKDIR}

In [ ]:
# Installation. torch, torchaudio and torchcodec are the builds that extracted
# the original features: the pinned versions for CUDA 13.0. If anything pinned
# is missing or has another version, everything is installed in one pass and
# the cell always stops for a restart: a kernel never continues after pip has
# run in it (the runtime has imported numpy before this cell, and a module
# that is already loaded stays loaded when pip replaces it on disk).
# On the run after the restart nothing is installed: the cell checks the
# installed versions and the versions of the modules that the kernel loads.
import importlib
import importlib.metadata
import re
import sys

TORCH_INDEX = "https://download.pytorch.org/whl/cu130"
TORCH_FAMILY = ("torch", "torchaudio", "torchcodec")          # installed from TORCH_INDEX
INSTALLED_ONCE = os.path.join(os.path.dirname(WORKDIR), "installed_once.txt")
OTHERS = os.path.join(os.path.dirname(WORKDIR), "requirements_without_torch.txt")
CONSTRAINT = os.path.join(os.path.dirname(WORKDIR), "torch_constraint.txt")

with open(os.path.join(WORKDIR, "requirements.txt"), encoding="utf-8") as fh:
    requirements = [line.strip() for line in fh if line.strip() and not line.lstrip().startswith("#")]
pins = dict(line.split("==") for line in requirements if "==" in line)
BUILDS = {name: pins[name] + "+" + TORCH_INDEX.rsplit("/", 1)[1] for name in TORCH_FAMILY}


def installed(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


def not_as_pinned():
    """{package: (installed, wanted)} for everything that is missing or has another version."""
    found = {}
    for line in requirements:
        name = re.split(r"[=<>!~ ]", line, maxsplit=1)[0]
        have, want = installed(name), BUILDS.get(name, pins.get(name))
        same = have is not None and (want is None or (have if name in BUILDS else have.split("+")[0]) == want)
        if not same:
            found[name] = (have, want)
    if installed("ssl-silence") is None:
        found["ssl-silence"] = (None, "this repository")
    return found


different = not_as_pinned()
if different:
    assert not os.path.exists(INSTALLED_ONCE), (
        f"the libraries were installed once already and these are still not as pinned: {different}. "
        "Stop: do not restart again")
    imported = sorted(name for name in (*TORCH_FAMILY, "transformers", "ssl_silence") if name in sys.modules)
    assert not imported, (f"{imported} were imported before the installation: "
                          "Runtime > Restart session, then run the notebook again from the top")
    with open(OTHERS, "w", encoding="utf-8") as fh:
        for line in requirements:
            if re.split(r"[=<>!~ ]", line, maxsplit=1)[0] not in TORCH_FAMILY:
                print(line, file=fh)
    with open(CONSTRAINT, "w", encoding="utf-8") as fh:
        for name in TORCH_FAMILY:
            print(f"{name}=={BUILDS[name]}", file=fh)
    # The build is part of the pin (torch==2.11.0+cu130): for a pin without it pip
    # keeps another build of the same version that the runtime ships (+cpu).
    FAMILY_PINS = " ".join(f"{name}=={BUILDS[name]}" for name in TORCH_FAMILY)
    # torchvision of the runtime is built for its own torch and breaks imports
    # with another one; this package does not use it.
    !pip uninstall -q -y torchvision
    !pip install -q {FAMILY_PINS} --index-url {TORCH_INDEX}
    !pip install -q -r {OTHERS} -c {CONSTRAINT}
    !pip install -q --no-deps -e .
    importlib.invalidate_caches()
    # Before the restart: everything on disk is as pinned, so the next run installs nothing.
    different = not_as_pinned()
    assert not different, f"pip did not install these as pinned: {different}: stop"
    with open(INSTALLED_ONCE, "w", encoding="utf-8") as fh:
        print("the pinned libraries were installed in this runtime", file=fh)
    # pip has run in this kernel: it always stops here.
    raise RuntimeError("The pinned libraries were installed. "
                       "Runtime > Restart session, then run the notebook again from the top")

# Nothing was installed in this run. The modules that this kernel loads must
# be the pinned versions too, not only the files on disk.
LOADED = {"numpy": "numpy", "scipy": "scipy", "pandas": "pandas", "scikit-learn": "sklearn",
          "torch": "torch", "torchaudio": "torchaudio", "transformers": "transformers"}
for name, module in LOADED.items():
    loaded = importlib.import_module(module).__version__
    assert loaded.split("+")[0] == pins[name], (
        f"{module} {loaded} is loaded in this kernel and the pin is {pins[name]}: it was imported before pip "
        "replaced it. Runtime > Restart session, then run the notebook again from the top")

import torch
import torchaudio
import transformers

print("loaded:", {name: sys.modules[module].__version__ for name, module in LOADED.items()})
print("torch", torch.__version__, "| CUDA of the build:", torch.version.cuda)
print("torchaudio", torchaudio.__version__, "| torchcodec", installed("torchcodec"),
      "| transformers", transformers.__version__)
if os.path.join(WORKDIR, "src") not in sys.path:
    sys.path.insert(0, os.path.join(WORKDIR, "src"))   # a running kernel does not see a new editable install

In [ ]:
import subprocess
import sys

import pandas as pd

from ssl_silence import config as cfg
from ssl_silence import provenance, verify

pd.set_option("display.max_rows", 200, "display.width", 200)
HEAD = provenance.git_commit()     # the full hash, with "-dirty" if a tracked file was changed
assert HEAD.startswith(COMMIT) and not HEAD.endswith("-dirty"), (HEAD, COMMIT)
print("commit:", HEAD)
print("root:", cfg.ROOT)

### Runtime

In [ ]:
report = provenance.environment_report()
display(report)
result = subprocess.run([sys.executable, "-m", "pytest", "-q", "-rfs", "tests/test_environment.py"],
                        env=dict(os.environ, SSL_SILENCE_REQUIRE_REFERENCE="1"), capture_output=True, text=True)
print(result.stdout + result.stderr)
assert result.returncode == 0, "the installed libraries are not the pinned ones"

### New root (first write to Drive)

Creates the folders of the new root. Nothing under the original folders is
written, moved or deleted by any notebook of this repository.

The free space is the quota of the Google account, read from the Drive API
(limit minus usage); the cell stops if it cannot be read or is below the
required amount. The free space of the local disk of the runtime, where the
datasets are unpacked, is printed as information.

In [ ]:
for folder in (cfg.ROOT, cfg.checks_dir(), cfg.migration_dir(), cfg.datasets_dir()):
    cfg.makedirs(folder)
# The quota of the Google account is read from the Drive API (this asks for
# access to the account); the mounted folder does not report it.
QUOTA = cfg.drive_quota()
print("Google Drive: {usage_gb:.1f} GB used of {limit_gb:.1f} GB, {free_gb:.1f} GB free".format(**QUOTA),
      "(GB = 10^9 bytes; required: %d GB free)" % cfg.MIN_FREE_GB)
cfg.require_free_space(quota=QUOTA)
print("local disk of the runtime: %.1f GB free" % cfg.local_free_space_gb())

### Mapping file

`legacy_map.yaml` names the original folders, files and labels. It is kept on
Drive only, at `<root>/migration/legacy_map.yaml`, and must have no empty
value (`verify.load_map` stops otherwise).

On the first run this cell stops: upload `legacy_map.yaml` to
`<root>/migration/` (the previous cell created it), then run the rest.

In [ ]:
MAP_PATH = os.path.join(cfg.migration_dir(), "legacy_map.yaml")
assert os.path.exists(MAP_PATH), f"upload legacy_map.yaml to {MAP_PATH}"
legacy = verify.load_map(MAP_PATH)
FORBIDDEN = legacy["forbidden"]
CHECKS = cfg.checks_dir()
checks = {}


def record(name, table, stop=True):
    """Write a check table to Drive, show it, and stop if a row fails."""
    path = verify.write_check(table, name, CHECKS, FORBIDDEN)
    checks[name] = table
    display(table)
    if stop and "ok" in table.columns:
        verify.require_ok(table, name)
    return path

In [ ]:
record("environment", provenance.environment_report(cfg.ROOT, quota=QUOTA), stop=False)

### Original analysis code (read only)

The parity tests import the original code. Its repository and commit are
named in the mapping file (`original_code`).

In [ ]:
ORIGINAL = "/content/original_code"
original = legacy["original_code"]
assert token, ("the original analysis code is not public: add the Colab secret GITHUB_TOKEN "
               "and grant this notebook access")
clone(original["repository"], ORIGINAL)
git("-C", ORIGINAL, "checkout", "-q", str(original["commit"]))
assert provenance.git_commit(ORIGINAL).startswith(str(original["commit"])), provenance.git_commit(ORIGINAL)
os.environ["SSL_SILENCE_ORIGINAL_SRC"] = os.path.join(ORIGINAL, "src")

### ASVspoof 2019 LA on the local disk, and the file lists

The archive is read from its original place and unpacked to the local disk of
the runtime. The official ASV score files are taken out of the same archive.

In [ ]:
from ssl_silence import datasets, metrics

LA_ZIP = verify.legacy_path(legacy, legacy["files"]["asvspoof_zip"])
datasets.ensure_asvspoof(LA_ZIP)
ASV_SCORES = "/content/asv_scores"
metrics.extract_asv_scores(LA_ZIP, ASV_SCORES)

file_lists = {split: datasets.items(split) for split in ("train", "dev", "eval")}
rows = [{"split": split, **datasets.check_items(file_list, split), "ok": True}
        for split, file_list in file_lists.items()]
record("file_lists_asvspoof", pd.DataFrame(rows))

### The test suite on this runtime

All tests, with:

* the parity tests against the original code;
* the tandem t-DCF reference on the saved evaluation scores, required (`SSL_SILENCE_REQUIRE_REFERENCE=1`);
* audio loading, windows and trimming on real files (five files per split, fixed seed), compared with `==`.

The output of pytest is captured, printed below and written to `/content/pytest_output.txt`; its last line is recorded as the check `test_suite`, and a failing test stops the notebook after that. The parity record of this runtime is copied to the checks folder.

In [ ]:
REAL_FILES = "/content/real_files.csv"
sample = pd.concat([file_list.sample(n=5, random_state=0) for file_list in file_lists.values()])
sample[["path"]].to_csv(REAL_FILES, index=False)

env = dict(os.environ, SSL_SILENCE_ASV_SCORES=ASV_SCORES, SSL_SILENCE_REQUIRE_REFERENCE="1",
           SSL_SILENCE_REAL_FILES=REAL_FILES,
           SSL_SILENCE_REAL_FILES_RECORD="/content/real_files_record.csv")
result = subprocess.run([sys.executable, "-m", "pytest", "-q", "-rfs"], env=env, capture_output=True, text=True)
PYTEST_OUTPUT = "/content/pytest_output.txt"
with open(PYTEST_OUTPUT, "w", encoding="utf-8") as fh:
    fh.write(result.stdout + result.stderr)
print(result.stdout + result.stderr)
print("the whole output is also in", PYTEST_OUTPUT)
lines = result.stdout.strip().splitlines()
record("test_suite", pd.DataFrame([{"tests": "all", "returncode": result.returncode,
                                    "summary": lines[-1].strip("= ") if lines else "",
                                    "ok": result.returncode == 0}]))

In [ ]:
parity = pd.read_csv("results/checks/parity_original_code.csv")
record("parity_original_code_colab", parity.assign(ok=parity["identical"]))
compared = pd.read_csv("/content/real_files_record.csv")
record("real_files_compared", compared.assign(split=sample["split"].to_numpy()), stop=False)

### Audio libraries of the original extraction, and original sampling rates

What the original feature shards record (which library read the audio, and
the library versions), and how many files were not recorded at 16 kHz
according to the original measurement tables.

In [ ]:
from ssl_silence import migration

source = verify.legacy_path(legacy, legacy["features"]["wavlm-base"]["unpadded"]["clean"])
record("original_shard_provenance", migration.shard_provenance(source), stop=False)

folder = verify.legacy_path(legacy, legacy["files"]["measurements_dir"])
tables = {split: pd.read_csv(os.path.join(folder, f"{split}.csv"), usecols=["sr_orig"])
          for split in cfg.SPLITS}
record("original_sampling_rates", migration.other_sampling_rates(tables), stop=False)

### In-the-Wild: the revision to pin

Reads the current commit of the dataset repository; nothing is downloaded.
Write the printed commit into `config.ITW_REVISION`, commit, and run
`notebooks/02_migrate_wavlm_features.ipynb` at that commit: it downloads the
archive once from the pinned revision and prints its sha256 for
`config.ITW_ZIP_SHA256`.

In [ ]:
revision = provenance.dataset_revision(cfg.ITW_REPOSITORY)
print("In-the-Wild repository:", cfg.ITW_REPOSITORY)
print("current commit        :", revision)
print("config.ITW_REVISION   :", cfg.ITW_REVISION)
record("in_the_wild_revision", pd.DataFrame([{"repository": cfg.ITW_REPOSITORY, "commit": revision,
                                              "pinned": str(cfg.ITW_REVISION)}]), stop=False)

### Summary

In [ ]:
summary = verify.summarise_checks({name: table for name, table in checks.items() if "ok" in table.columns})
display(summary)
print("check tables written to", CHECKS)
assert summary["ok"].all()